In [1]:


import numpy as np 
import pandas as pd 


In [2]:
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for pyg_lib

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip -q install tqdm



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
from tqdm.auto import tqdm


/Users/abdelrahmanelkhayat/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import SentenceTransformer
print("OK - imports succeeded")


OK - imports succeeded


In [6]:
!pip -q install -U sentence-transformers datasets accelerate



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
# ============================================================
# Fine-tune BAAI/bge-base-en-v1.5 on JSONL (view1, view2) pairs
# With a real tqdm progress bar and manual training loop.
# ============================================================


import os, json, math, random, re, time
from typing import Any, List

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import get_linear_schedule_with_warmup
from sentence_transformers import SentenceTransformer, InputExample, losses

# ---------------------------
# 1) Configuration
# ---------------------------
MODEL_NAME = "BAAI/bge-base-en-v1.5"

# IMPORTANT: set your train path
TRAIN_JSONL = "Combined_synthetic.jsonl"


# Start conservative; scale up after it runs
BATCH_SIZE = 32            
MAX_SEQ_LENGTH = 256      
EPOCHS = 1                
LR = 2e-5
WARMUP_RATIO = 0.1
FP16 = True               
GRAD_ACCUM = 2             
SEED = 42

USE_PREFIX = False
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "


In [10]:


# ---------------------------
# 2) Reproducibility
# ---------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------
# 3) Text cleaning utilities
# ---------------------------
def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    return x if isinstance(x, str) else str(x)

def extract_text_maybe_json(s: str) -> str:
    """
    If a field is a JSON-encoded string (e.g., '{ "...": "text" }'),
    try to parse it and extract useful text.
    """
    s = _safe_str(s).strip()
    if not s:
        return ""

    if s.startswith("{") and s.endswith("}"):
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                
                if "" in obj and isinstance(obj[""], str) and obj[""].strip():
                    return obj[""].strip()

                # Otherwise join string values
                parts = []
                for v in obj.values():
                    if isinstance(v, str) and v.strip():
                        parts.append(v.strip())
                if parts:
                    return "\n".join(parts).strip()
        except Exception:
            pass

    return s

def normalize_text(s: str) -> str:
    s = extract_text_maybe_json(s)
    # Normalize whitespace
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def maybe_prefix(v1: str, v2: str):
    if not USE_PREFIX:
        return v1, v2
    return QUERY_PREFIX + v1, PASSAGE_PREFIX + v2

# ---------------------------
# 4) Load training pairs
# ---------------------------
def load_pairs_from_jsonl(path: str) -> List[InputExample]:
    examples: List[InputExample] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            v1 = normalize_text(obj.get("view1", ""))
            v2 = normalize_text(obj.get("view2", ""))

            if not v1 or not v2:
                continue

            v1, v2 = maybe_prefix(v1, v2)
            examples.append(InputExample(texts=[v1, v2]))

    return examples

assert os.path.exists(TRAIN_JSONL), f"TRAIN_JSONL not found: {TRAIN_JSONL}"
train_examples = load_pairs_from_jsonl(TRAIN_JSONL)

print("Train pairs loaded:", len(train_examples))
assert len(train_examples) > 0, "No training examples found. Check your JSONL keys/path."

# ---------------------------
# 5) Load model (if this hangs, Kaggle internet is likely off)
# ---------------------------
t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = MAX_SEQ_LENGTH
model.to(device)
print(f"Loaded model in {time.time()-t0:.1f}s")

# ---------------------------
# 6) DataLoader with ST's smart batching collate
# ---------------------------
train_dataloader = DataLoader(
    train_examples,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=model.smart_batching_collate
)

print("Batches per epoch:", len(train_dataloader))
assert len(train_dataloader) > 0, "Dataloader has 0 batches. Increase data or reduce batch size."

# ---------------------------
# 7) Loss + Optimizer + Scheduler
# ---------------------------
train_loss = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = len(train_dataloader)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=(FP16 and device == "cuda"))

print(f"Steps/epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print("Starting training...")

# ---------------------------
# 8) Training loop with tqdm progress bar
# ---------------------------
global_step = 0
model.train()

for epoch in range(EPOCHS):
    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=True
    )

    for step, batch in enumerate(progress_bar):
        sentence_features, labels = batch

        # Move batch to device
        for sf in sentence_features:
            for k in sf:
                sf[k] = sf[k].to(device)

        # Forward + backward
        with torch.cuda.amp.autocast(enabled=(FP16 and device == "cuda")):
            loss_val = train_loss(sentence_features, labels)
            loss_scaled = loss_val / GRAD_ACCUM

        scaler.scale(loss_scaled).backward()

        # Optim step every GRAD_ACCUM steps
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        global_step += 1

        # Update progress bar postfix every step
        progress_bar.set_postfix(
            loss=f"{loss_val.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

print("Training complete.")

Device: cpu
Train pairs loaded: 2377


/var/folders/qd/qtdqglvs7tz_v0q305tw1dm00000gn/T/ipykernel_1987/4090043788.py:138: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(FP16 and device == "cuda"))


Loaded model in 2.3s
Batches per epoch: 74
Steps/epoch: 74
Total steps: 74
Warmup steps: 7
Starting training...


Epoch 1/1:   0%|          | 0/74 [00:00<?, ?it/s]/var/folders/qd/qtdqglvs7tz_v0q305tw1dm00000gn/T/ipykernel_1987/4090043788.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(FP16 and device == "cuda")):
Epoch 1/1: 100%|██████████| 74/74 [1:01:21<00:00, 49.75s/it, loss=0.0183, lr=1.10e-05] 

Training complete.


In [20]:



# Pick a random training pair and compute cosine similarity
idx = random.randint(0, len(train_examples) - 1)
a, b = train_examples[idx].texts

ea = model.encode([a], normalize_embeddings=True)
eb = model.encode([b], normalize_embeddings=True)
sim = float(ea @ eb.T)

print("\nSanity check cosine similarity (positive pair):", sim)
print("\n--- view1 ---\n", a[:500])
print("\n--- view2 ---\n", b[:500])



Sanity check cosine similarity (positive pair): 0.7351040244102478

--- view1 ---
 In the midst of an interstellar conflict, a fleet of human spacecraft engages in a desperate battle against an alien empire known as the Xathar Dominion. Captain Elena Voss, commanding the flagship *Resolute*, devises a risky plan to disable the Xathar's central warship, which controls their fleet through a network of quantum-linked drones. A covert team infiltrates the warship, planting explosives near its core while evading detection from Xathar soldiers. Meanwhile, the *Resolute* sustains hea

--- view2 ---
 Amid a cosmic struggle, a coalition of planetary forces fights to fend off an invasion by the alien Krylex Collective. Commander Arlen Drake, leading the battleship *Vanguard*, formulates a daring strategy to cripple the Krylex's command vessel, which orchestrates its fleet using an advanced neural synchronization array. A specialized strike unit infiltrates the command ship, planting sabotage de

/var/folders/qd/qtdqglvs7tz_v0q305tw1dm00000gn/T/ipykernel_1987/3476569741.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sim = float(ea @ eb.T)


In [21]:
import sys
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import numpy as np
def evaluate(labeled_data_path, embedding_lookup):
    df = pd.read_json(labeled_data_path, lines=True)

    # Map texts to embeddings
    df["anchor_embedding"] = df["anchor_text"].map(embedding_lookup)
    df["a_embedding"] = df["text_a"].map(embedding_lookup)
    df["b_embedding"] = df["text_b"].map(embedding_lookup)

    # Look up cosine similarities
    df["sim_a"] = df.apply(
        lambda row: cos_sim(row["anchor_embedding"], row["a_embedding"]), axis=1
    )
    df["sim_b"] = df.apply(
        lambda row: cos_sim(row["anchor_embedding"], row["b_embedding"]), axis=1
    )

    # Predict and calculate accuracy
    df["predicted_text_a_is_closer"] = df["sim_a"] > df["sim_b"]
    accuracy = (df["predicted_text_a_is_closer"] == df["text_a_is_closer"]).mean()
    return accuracy


In [26]:
# Select baseline method
baseline = "sbert"  # or "random"
data = pd.read_json("dev_track_b.jsonl", lines=True)

embeddings = model.encode(data["text"], show_progress_bar=True)
embedding_lookup = dict(zip(data["text"], embeddings))
accuracy = evaluate("dev_track_a.jsonl", embedding_lookup)
print(f"Accuracy: {accuracy:.3f}")


Batches: 100%|██████████| 15/15 [00:13<00:00,  1.14it/s]

Accuracy: 0.645
